In [77]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [78]:
## Load the ANN trained model, scaler pickle, onehot
model= load_model('model.h5')

In [79]:
##Load envcoders and scaler
with open('one_hot_encoder_geography.pkl','rb') as file:
    label_encoder_geo = pickle.load(file)

with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)

In [80]:
input_data={
    'CreditScore':600,
    'Geography':'France',
    'Gender':'Male',
    'Age':40,
    'Tenure':3,
    'Balance':600000,
    'NumOfProducts':2,
    'HasCrCard':1,
    'IsActiveMember':1,
    'EstimatedSalary':50000,
}

In [81]:
#One hot encoding for Geography
geo_encoded = label_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=label_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

c:\Users\pratik.narawade\OneDrive - PharmaACE\Production_Team\DL\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [82]:
pd.DataFrame([input_data])

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,600000,2,1,1,50000


In [83]:
## ENcode categorical variables Gender so it will be in the same format as the training data
input_data['Gender'] = label_encoder_gender.transform([input_data['Gender']])[0]
input_data


{'CreditScore': 600,
 'Geography': 'France',
 'Gender': 1,
 'Age': 40,
 'Tenure': 3,
 'Balance': 600000,
 'NumOfProducts': 2,
 'HasCrCard': 1,
 'IsActiveMember': 1,
 'EstimatedSalary': 50000}

In [84]:
##Concatenate the one hot encoded geography with the rest of the input data with droppping Geaography column
input_data_encoded = pd.concat([pd.DataFrame([input_data]).drop(columns=['Geography']), geo_encoded_df], axis=1)
input_data_encoded

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,600000,2,1,1,50000,1.0,0.0,0.0


In [85]:
##Scaling the input data
input_data_scaled = scaler.transform(input_data_encoded)
input_data_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349,  8.38812313,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [87]:
## Predicting the output using the trained model
prediction = model.predict(input_data_scaled)

1/1 [==============================] - 0s 42ms/step


In [90]:
## Get probability of churn
churn_probability = prediction[0][0]
churn_probability

0.40088156

In [91]:
if churn_probability > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is not likely to churn.")
    

The customer is not likely to churn.
